# CMT Rare Variant Pipeline - UK Biobank WGS (500k)
**DNAnexus Research Analysis Platform (RAP) - JupyterLab notebook**


### Pipeline stages
1. **Stage 0** - Setup: install dependencies, configure RAP project paths
2. **Stage 0b** - CMT participant identification: query UKB phenotype DB for ICD-10 CMT codes
3. **Stage 1** - MAF extraction: parallel SAK jobs computing per-gene MAF from whole WGS cohort
4. **Stage 2** - Combine & filter MAF: merge outputs, keep MAF < 0.001
5. **Stage 3** - CMT cohort extraction: filter to diagnosed EIDs, extract gene region VCFs
6. **Stage 4** - Carrier identification: parse VCFs, determine zygosity, build carrier table
7. **Stage 5** - VEP annotation of CMT cohort VCF (carried variants only)
8. **Stage 6** - Pathogenic filtering: keep variants with CLIN_SIG = pathogenic / likely_pathogenic
9. **Stage 7** - Generate per-chromosome variant lists from pathogenic variants only
10. **Stage 8** - Whole-population extraction: extract pathogenic variants across all 500k participants
11. **Stage 9** - Download, combine into single VCF, and inspect results



---
## Stage 0 - Setup & Configuration


In [ ]:
%%bash
pip install dxpy --quiet
pip install polars


In [ ]:
import dxpy
import subprocess
import pandas as pd
import polars as pl
import glob
import os
import time

# ── Configure your project ──────────────────────────────────────────────────
PROJECT_ID = dxpy.PROJECT_CONTEXT_ID
print(f'Using RAP project: {PROJECT_ID}')

# ── Output folder on RAP — NO trailing slash (prevents double-slash in paths) ─
OUTPUT_FOLDER = '/CMT_WGS_analysis'

# ── WGS PLINK2 data path ─────────────────────────────────────────────────────
WGS_PLINK_PATH = '/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]'

# ── Gene coordinate input files ──────────────────────────────────────────────
GENE_LOC_FILE   = '/CMT_WGS_analysis/input_data/cmt_gene_locations.csv'
GENE_LOC_X_FILE = '/CMT_WGS_analysis/input_data/cmt_gene_location_x.csv'

# ── EID files generated in Stage 0b ─────────────────────────────────────────
EID_FILE       = 'diagnosed_eid.txt'
EID_PLINK_FILE = 'diagnosed_eid_plink.txt'

# ── Instance type ────────────────────────────────────────────────────────────
INSTANCE_TYPE = 'mem2_ssd2_x8'

print(f'Output folder: {OUTPUT_FOLDER}')
print('Configuration set.')


In [ ]:
def submit_sak_job(name, cmd, output_folder, instance=INSTANCE_TYPE):
    dx_cmd = [
        'dx', 'run', 'swiss-army-knife',
        '--instance-type', instance,
        '--folder', output_folder,
        '--name', name,
        '-icmd=' + cmd,
        '-y', '--brief'
    ]
    result = subprocess.run(dx_cmd, capture_output=True, text=True, check=True)
    job_id = result.stdout.strip()
    print(f'  Submitted: {name} -> {job_id}')
    return job_id


def wait_for_jobs(job_ids, poll_interval=30):
    print(f'Waiting for {len(job_ids)} jobs...')
    pending = set(job_ids)
    failed  = []
    while pending:
        still_running = set()
        for jid in list(pending):
            state = dxpy.describe(jid)['state']
            if state == 'done':
                print(f'  DONE  {jid}')
            elif state in ('failed', 'terminated'):
                print(f'  FAIL  {jid} ({state})')
                failed.append(jid)
            else:
                still_running.add(jid)
        pending = still_running
        if pending:
            time.sleep(poll_interval)
    if failed:
        raise RuntimeError(f'Jobs failed: {failed}')
    print('All jobs completed successfully.')


def read_vcf(filepath):
    """Read a VCF file, correctly skipping ## meta-lines, using Polars for speed."""
    with open(filepath, 'r') as f:
        skip = sum(1 for line in f if line.startswith('##'))
    df = pl.read_csv(filepath, separator='\t', skip_rows=skip, infer_schema_length=0)
    df.columns = [c.lstrip('#') for c in df.columns]
    return df.to_pandas()


print('Helper functions defined.')


---
## Stage 0b - Identify CMT-Diagnosed Participants

Queries the UK Biobank phenotype database (via Spark/dxdata) for participants
with a CMT ICD-10 diagnosis code in field `p41270` (Hospital Episode Statistics).

- G60.0 CMT ICD-10 code used


In [ ]:
import pyspark
import dxpy
import dxdata
import csv

sc    = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_database_name = dxpy.find_one_data_object(
    classname='database', name='app*', folder='/',
    name_mode='glob', describe=True
)['describe']['name']

dispensed_dataset_id = dxpy.find_one_data_object(
    typename='Dataset', name='app*.dataset',
    folder='/', name_mode='glob'
)['id']

spark.sql('USE ' + dispensed_database_name)

dataset     = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset['participant']

print(f'Connected to database: {dispensed_database_name}')
print(f'Dataset ID: {dispensed_dataset_id}')


In [ ]:
CMT_ICD_CODES = {'G600'}

field_names = ['eid', 'p41270']
engine = dxdata.connect(dialect='hive+pyspark')
df = participant.retrieve_fields(names=field_names, coding_values='raw', engine=engine)

data_dicts = [
    {'eid': row.eid, 'p41270': row.p41270}
    for row in df.rdd.toLocalIterator()
]

csv_file_path = 'UKB_icd_date.csv'
with open(csv_file_path, 'w', newline='') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=['eid', 'p41270'])
    writer.writeheader()
    for data_dict in data_dicts:
        writer.writerow(data_dict)

print(f'Raw ICD extract saved to {csv_file_path} ({len(data_dicts):,} participants total)')


In [ ]:
cmt_eids = []
for row in data_dicts:
    icd_list = row['p41270']
    if icd_list is None:
        continue
    if isinstance(icd_list, str):
        icd_list = [code.strip() for code in icd_list.strip('[]').split(',')]
    if any(str(code).strip() in CMT_ICD_CODES for code in icd_list):
        cmt_eids.append(str(row['eid']))

print(f'CMT-diagnosed participants identified: {len(cmt_eids)}')

with open(EID_FILE, 'w') as f:
    for eid in cmt_eids:
        f.write(eid + '\n')
print(f'Written: {EID_FILE} ({len(cmt_eids)} EIDs)')

with open(EID_PLINK_FILE, 'w') as f:
    for eid in cmt_eids:
        f.write(f'{eid} {eid}\n')
print(f'Written: {EID_PLINK_FILE} ({len(cmt_eids)} individuals in FID IID format)')


In [ ]:
dx_eid_path       = f'{OUTPUT_FOLDER}/inputs/diagnosed_eid.txt'
dx_eid_plink_path = f'{OUTPUT_FOLDER}/inputs/diagnosed_eid_plink.txt'

subprocess.run(['dx', 'upload', EID_FILE,       '--destination', dx_eid_path,       '--brief'], check=True)
subprocess.run(['dx', 'upload', EID_PLINK_FILE, '--destination', dx_eid_plink_path, '--brief'], check=True)
print(f'Uploaded EID files to RAP: {dx_eid_path}')


---
## Stage 1 - MAF Extraction across Whole WGS Cohort

One SAK job per chromosome (autosomes + X), all running in parallel.


In [ ]:
subprocess.run(['dx', 'download', GENE_LOC_FILE,   '-o', 'cmt_gene_regions.csv',  '-f'], check=True)
subprocess.run(['dx', 'download', GENE_LOC_X_FILE, '-o', 'cmt_gene_regions_x.csv', '-f'], check=True)

gene_location   = pd.read_csv('cmt_gene_regions.csv')
gene_location_x = pd.read_csv('cmt_gene_regions_x.csv')
print(f'Loaded {len(gene_location)} autosomal CMT gene entries')
print(f'Loaded {len(gene_location_x)} X-chromosome CMT gene entries')
print(gene_location.head())


In [ ]:
chromosomes = list(range(1, 23))
maf_job_ids = []

for chrom in chromosomes:
    temp = gene_location[gene_location['chrom'] == chrom]
    if temp.empty:
        continue

    pfile_base    = f'{WGS_PLINK_PATH}/ukb24308_c{chrom}_b0_v1'
    per_gene_cmds = []
    for _, row in temp.iterrows():
        g = row['gene']
        s = int(row['start'])
        e = int(row['end'])
        per_gene_cmds.append(
            f'plink2 --pfile "/mnt/project{pfile_base}" '
            f'--no-psam-pheno '
            f'--chr {chrom} --from-bp {s} --to-bp {e} '
            f'--max-maf 0.001 --freq --out {g}_freq_out'
        )

    job_id = submit_sak_job(
        name          = f'CMT_WGS_MAF_chr{chrom}',
        cmd           = ' ; '.join(per_gene_cmds),
        output_folder = f'{OUTPUT_FOLDER}/maf_freq',
        instance      = INSTANCE_TYPE
    )
    maf_job_ids.append(job_id)

print(f'Submitted {len(maf_job_ids)} autosomal MAF jobs.')


In [ ]:
# chrX MAF extraction
per_gene_cmds_x = []
for _, row in gene_location_x.iterrows():
    g = row['gene']
    s = int(row['start'])
    e = int(row['end'])
    per_gene_cmds_x.append(
        f'plink2 --pfile "/mnt/project{WGS_PLINK_PATH}/ukb24308_cX_b0_v1" '
        f'--no-psam-pheno '
        f'--chr X --from-bp {s} --to-bp {e} '
        f'--max-maf 0.001 --freq --out {g}_freq_out'
    )

job_id_x_maf = submit_sak_job(
    name          = 'CMT_WGS_MAF_chrX',
    cmd           = ' ; '.join(per_gene_cmds_x),
    output_folder = f'{OUTPUT_FOLDER}/maf_freq',
    instance      = INSTANCE_TYPE
)
maf_job_ids.append(job_id_x_maf)
print(f'Submitted chrX MAF job: {job_id_x_maf}')


In [ ]:
wait_for_jobs(maf_job_ids)


---
## Stage 2 - Combine and Filter MAF Results


In [ ]:
os.makedirs('maf_outputs', exist_ok=True)
subprocess.run(
    ['dx', 'download', f'{OUTPUT_FOLDER}/maf_freq/*.afreq', '-o', 'maf_outputs/', '-f'],
    check=True
)

def read_afreq(filepath):
    df = pd.read_csv(filepath, sep='\t')
    df.columns = [c.lstrip('#') for c in df.columns]
    df = df.rename(columns={'ALT_FREQS': 'MAF', 'CHROM': 'CHR'})
    return df

afreq_files    = glob.glob('maf_outputs/*.afreq')
print(f'Found {len(afreq_files)} .afreq files')

freq_data_list = [read_afreq(f) for f in afreq_files]
combined_freq  = pd.concat(freq_data_list, ignore_index=True)

filtered_freq_data = combined_freq[combined_freq['MAF'] < 0.001].copy()
print(f'Total variants before MAF filter: {len(combined_freq):,}')
print(f'Total variants after  MAF filter: {len(filtered_freq_data):,}')
filtered_freq_data.head()


---
## Stage 3 - CMT Cohort Gene Region Extraction

One SAK job per chromosome, all running in parallel.


In [ ]:
dx_eid_path       = f'{OUTPUT_FOLDER}/inputs/diagnosed_eid.txt'
dx_eid_plink_path = f'{OUTPUT_FOLDER}/inputs/diagnosed_eid_plink.txt'
print(f'Using EID files: {dx_eid_plink_path}')


In [ ]:
cmt_extract_job_ids = []

for chrom in chromosomes:
    temp = gene_location[gene_location['chrom'] == chrom]
    if temp.empty:
        continue

    pfile_base = f'{WGS_PLINK_PATH}/ukb24308_c{chrom}_b0_v1'

    step1 = (
        f'plink2 --pfile "/mnt/project{pfile_base}" '
        f'--no-psam-pheno '
        f'--keep /mnt/project{dx_eid_plink_path} '
        f'--make-pgen --out chr{chrom}_cmt_filtered'
    )

    per_gene = []
    for _, row in temp.iterrows():
        g = row['gene']
        s = int(row['start'])
        e = int(row['end'])
        per_gene.append(
            f'plink2 --pfile chr{chrom}_cmt_filtered '
            f'--no-psam-pheno '
            f'--chr {chrom} --from-bp {s} --to-bp {e} '
            f'--recode vcf --out {g}_gene'
        )

    job_id = submit_sak_job(
        name          = f'CMT_WGS_Extract_chr{chrom}',
        cmd           = step1 + ' ; ' + ' ; '.join(per_gene),
        output_folder = f'{OUTPUT_FOLDER}/cmt_vcfs',
        instance      = INSTANCE_TYPE
    )
    cmt_extract_job_ids.append(job_id)

print(f'Submitted {len(cmt_extract_job_ids)} autosomal CMT extraction jobs.')


In [ ]:
# chrX extraction
step1_x = (
    f'plink2 --pfile "/mnt/project{WGS_PLINK_PATH}/ukb24308_cX_b0_v1" '
    f'--no-psam-pheno '
    f'--keep /mnt/project{dx_eid_plink_path} '
    f'--make-pgen --out chrX_cmt_filtered'
)

per_gene_x = []
for _, row in gene_location_x.iterrows():
    g = row['gene']
    s = int(row['start'])
    e = int(row['end'])
    per_gene_x.append(
        f'plink2 --pfile chrX_cmt_filtered '
        f'--no-psam-pheno '
        f'--chr X --from-bp {s} --to-bp {e} '
        f'--recode vcf --out {g}_gene'
    )

job_id_x_extract = submit_sak_job(
    name          = 'CMT_WGS_Extract_chrX',
    cmd           = step1_x + ' ; ' + ' ; '.join(per_gene_x),
    output_folder = f'{OUTPUT_FOLDER}/cmt_vcfs',
    instance      = INSTANCE_TYPE
)
cmt_extract_job_ids.append(job_id_x_extract)
print(f'Submitted chrX extraction job: {job_id_x_extract}')


In [ ]:
wait_for_jobs(cmt_extract_job_ids)


---
## Stage 4 - Carrier Identification and Count Table


In [ ]:
os.makedirs('cmt_vcfs', exist_ok=True)
subprocess.run(
    ['dx', 'download', f'{OUTPUT_FOLDER}/cmt_vcfs/*_gene.vcf', '-o', 'cmt_vcfs/', '-f'],
    check=True
)
vcf_files = glob.glob('cmt_vcfs/*_gene.vcf')
print(f'Downloaded {len(vcf_files)} gene VCF files')


In [ ]:
vcf_data_list       = [read_vcf(f) for f in vcf_files]
gene_vcf_unfiltered = pd.concat(vcf_data_list, ignore_index=True)
print(f'Total variants before MAF merge: {len(gene_vcf_unfiltered):,}')

# Inner merge keeps only variants with MAF < 0.001 in the whole cohort
gene_vcf = pd.merge(gene_vcf_unfiltered, filtered_freq_data[['ID']], on='ID', how='inner')
print(f'Total variants after  MAF merge: {len(gene_vcf):,}')


In [ ]:
# Filter variant list using locally-held MAF data (fast — ID join only)
# We download just the ID column from freq data, not the full VCFs
vcf_files = glob.glob('cmt_vcfs/*_gene.vcf')
print(f'Downloaded {len(vcf_files)} gene VCF files')

# Collect all variant IDs from VCF files without loading genotype columns
def get_vcf_ids(filepath):
    """Return a set of variant IDs from a VCF (fast — reads only the ID column)."""
    ids = []
    with open(filepath) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            parts = line.split('\t', 4)
            if len(parts) >= 3:
                ids.append(parts[2])
    return ids

all_vcf_ids = set()
for vf in vcf_files:
    all_vcf_ids.update(get_vcf_ids(vf))

# Intersect with MAF-filtered IDs
maf_filtered_ids = set(filtered_freq_data['ID'].dropna().unique())
retained_ids     = all_vcf_ids & maf_filtered_ids
print(f'Variant IDs in CMT VCFs:          {len(all_vcf_ids):>8,}')
print(f'MAF-filtered variant IDs:          {len(maf_filtered_ids):>8,}')
print(f'Variants passing MAF filter:       {len(retained_ids):>8,}')

# Save the retained ID list to disk for the SAK carrier job
with open('retained_variant_ids.txt', 'w') as f:
    f.write('\n'.join(sorted(retained_ids)))
print('Written retained_variant_ids.txt')


In [ ]:
# ── Stage 4: Carrier identification via SAK job ──────────────────────────────
#
# Running extract_carriers() locally melts a wide VCF (samples × variants) into
# a huge long-format table, which exhausts the JupyterLab kernel memory.
# Instead we upload the VCFs + ID filter to RAP and run the heavy work on a
# dedicated SAK worker with more RAM.

carrier_script = r"""
import sys, os, glob, csv

VCF_DIR    = 'cmt_vcfs'
ID_FILE    = 'retained_variant_ids.txt'
OUT_FILE   = 'gene_carriers.csv'

with open(ID_FILE) as fh:
    retained = set(line.strip() for line in fh if line.strip())
print(f'Loaded {len(retained):,} retained variant IDs', flush=True)

vcf_files = sorted(glob.glob(f'{VCF_DIR}/*_gene.vcf'))
print(f'Processing {len(vcf_files)} VCF files', flush=True)

FIELDS = ['CHROM','POS','ID','REF','ALT','zygosity','carrier']

def zygosity(gt_field):
    gt = str(gt_field).split(':')[0].replace('|','/')
    if gt in ('0/1','1/0'): return 'het'
    if gt == '1/1':         return 'hom'
    return None

rows_written = 0
with open(OUT_FILE, 'w', newline='') as out_fh:
    writer = csv.writer(out_fh)
    writer.writerow(FIELDS)

    for vcf_path in vcf_files:
        samples = None
        with open(vcf_path) as fh:
            for line in fh:
                if line.startswith('##'):
                    continue
                cols = line.rstrip('\n').split('\t')
                if line.startswith('#CHROM'):
                    # Header row — capture sample names (columns 9+)
                    samples = cols[9:]
                    continue
                # Data row
                chrom, pos, vid, ref, alt = cols[0], cols[1], cols[2], cols[3], cols[4]
                if vid not in retained:
                    continue
                gt_fields = cols[9:]
                for sample, gt_val in zip(samples, gt_fields):
                    zyg = zygosity(gt_val)
                    if zyg is None:
                        continue
                    writer.writerow([chrom, pos, vid, ref, alt, zyg, sample[:7]])
                    rows_written += 1

        print(f'  done: {os.path.basename(vcf_path)}  (cumulative rows: {rows_written:,})', flush=True)

print(f'Carrier table written: {OUT_FILE}  ({rows_written:,} rows)', flush=True)
"""

 #Upload the retained-ID filter list and the VCF directory
subprocess.run(
    ['dx', 'upload', 'retained_variant_ids.txt',
     '--destination', f'{OUTPUT_FOLDER}stage4/retained_variant_ids.txt', '--brief'],
    check=True
)


# Upload the script to RAP once
with open('extract_carriers.py', 'w') as f:
    f.write(carrier_script)

subprocess.run(
    ['dx', 'upload', 'extract_carriers.py',
     '--destination', f'{PROJECT_ID}:/projects/CMT_NashBio/stage4/extract_carriers.py', '--brief'],
    check=True
)

sak_cmd = (
    f'pip install pandas -q && '
    f'mkdir -p cmt_vcfs && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/cmt_vcfs/*_gene.vcf" -o cmt_vcfs/ -f && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/stage4/retained_variant_ids.txt" -o retained_variant_ids.txt -f && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/stage4/extract_carriers.py" -o extract_carriers.py -f && '
    f'python3 extract_carriers.py && '
    f'dx upload gene_carriers.csv --destination "{PROJECT_ID}:/projects/CMT_NashBio/stage4/gene_carriers.csv" --brief'
)


carrier_job_id = submit_sak_job(
    name          = 'CMT_WGS_CarrierExtraction',
    cmd           = sak_cmd,
    output_folder = f'{OUTPUT_FOLDER}stage4',
    instance      = INSTANCE_TYPE
)
wait_for_jobs([carrier_job_id])
print(f'Carrier extraction complete: {carrier_job_id}')


In [ ]:
# ── Stage 4: Carrier identification via SAK job ──────────────────────────────
#
# Running extract_carriers() locally melts a wide VCF (samples × variants) into
# a huge long-format table, which exhausts the JupyterLab kernel memory.
# Instead we upload the VCFs + ID filter to RAP and run the heavy work on a
# dedicated SAK worker with more RAM.

carrier_script = r"""
import sys, os, glob, csv

VCF_DIR    = 'cmt_vcfs'
ID_FILE    = 'retained_variant_ids.txt'
OUT_FILE   = 'gene_carriers.csv'

with open(ID_FILE) as fh:
    retained = set(line.strip() for line in fh if line.strip())
print(f'Loaded {len(retained):,} retained variant IDs', flush=True)

vcf_files = sorted(glob.glob(f'{VCF_DIR}/*_gene.vcf'))
print(f'Processing {len(vcf_files)} VCF files', flush=True)

FIELDS = ['CHROM','POS','ID','REF','ALT','zygosity','carrier']

def zygosity(gt_field):
    gt = str(gt_field).split(':')[0].replace('|','/')
    if gt in ('0/1','1/0'): return 'het'
    if gt == '1/1':         return 'hom'
    return None

rows_written = 0
with open(OUT_FILE, 'w', newline='') as out_fh:
    writer = csv.writer(out_fh)
    writer.writerow(FIELDS)

    for vcf_path in vcf_files:
        samples = None
        with open(vcf_path) as fh:
            for line in fh:
                if line.startswith('##'):
                    continue
                cols = line.rstrip('\n').split('\t')
                if line.startswith('#CHROM'):
                    # Header row — capture sample names (columns 9+)
                    samples = cols[9:]
                    continue
                # Data row
                chrom, pos, vid, ref, alt = cols[0], cols[1], cols[2], cols[3], cols[4]
                if vid not in retained:
                    continue
                gt_fields = cols[9:]
                for sample, gt_val in zip(samples, gt_fields):
                    zyg = zygosity(gt_val)
                    if zyg is None:
                        continue
                    writer.writerow([chrom, pos, vid, ref, alt, zyg, sample[:7]])
                    rows_written += 1

        print(f'  done: {os.path.basename(vcf_path)}  (cumulative rows: {rows_written:,})', flush=True)

print(f'Carrier table written: {OUT_FILE}  ({rows_written:,} rows)', flush=True)
"""

 #Upload the retained-ID filter list and the VCF directory
subprocess.run(
    ['dx', 'upload', 'retained_variant_ids.txt',
     '--destination', f'{OUTPUT_FOLDER}stage4/retained_variant_ids.txt', '--brief'],
    check=True
)


# Upload the script to RAP once
with open('extract_carriers.py', 'w') as f:
    f.write(carrier_script)

subprocess.run(
    ['dx', 'upload', 'extract_carriers.py',
     '--destination', f'{PROJECT_ID}:/projects/CMT_NashBio/stage4/extract_carriers.py', '--brief'],
    check=True
)

sak_cmd = (
    f'pip install pandas -q && '
    f'mkdir -p cmt_vcfs && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/cmt_vcfs/*_gene.vcf" -o cmt_vcfs/ -f && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/stage4/retained_variant_ids.txt" -o retained_variant_ids.txt -f && '
    f'dx download "{PROJECT_ID}:/projects/CMT_NashBio/stage4/extract_carriers.py" -o extract_carriers.py -f && '
    f'python3 extract_carriers.py && '
    f'rm -rf cmt_vcfs/ retained_variant_ids.txt extract_carriers.py && '  # cleanup
    f'dx upload gene_carriers.csv --brief'
)



carrier_job_id = submit_sak_job(
    name          = 'CMT_WGS_CarrierExtraction',
    cmd           = sak_cmd,
    output_folder = f'{OUTPUT_FOLDER}stage4',
    instance      = INSTANCE_TYPE
)
wait_for_jobs([carrier_job_id])
print(f'Carrier extraction complete: {carrier_job_id}')


In [ ]:
counts      = gene_carriers.groupby(['ID', 'zygosity']).size().reset_index(name='Freq')
counts_wide = counts.pivot(index='ID', columns='zygosity', values='Freq').reset_index()
counts_wide.columns.name = None
counts_wide = counts_wide.fillna(pd.NA)

carrier_info = gene_carriers.drop(columns=['zygosity', 'carrier']).drop_duplicates()
final_counts = pd.merge(counts_wide, carrier_info, on='ID', how='left')

print(f'Final variant count table: {len(final_counts)} variants')
final_counts.head()


---
## Stage 5 - VEP Annotation of CMT Cohort VCF

Filters `gene_vcf` to only variants actually carried (present in `final_counts`),
then installs VEP locally and runs annotation in the JupyterLab session.

> Run the two bash cells once per session to install VEP. On subsequent runs,
> skip straight to the Python cell that writes `cmt_cohort.vcf`.


In [ ]:
# Filter to carried variants only before writing VEP input
carried_ids      = set(final_counts['ID'].dropna().unique())
gene_vcf_carried = gene_vcf[gene_vcf['ID'].isin(carried_ids)].copy()

print(f'Variants in gene_vcf (MAF-filtered):     {len(gene_vcf):>8,}')
print(f'Variants in final_counts (carried):       {len(final_counts):>8,}')
print(f'Variants passing carried filter:          {len(gene_vcf_carried):>8,}')

# Write VCF with correct #CHROM header for VEP
with open('cmt_cohort.vcf', 'w') as f:
    f.write('##fileformat=VCFv4.2\n')
    f.write('##source=CMT_WGS_pipeline\n')
    gene_vcf_out = gene_vcf_carried.copy()
    gene_vcf_out.columns = ['#CHROM' if c == 'CHROM' else c for c in gene_vcf_out.columns]
    gene_vcf_out.to_csv(f, sep='\t', index=False)

print(f'Written cmt_cohort.vcf: {len(gene_vcf_carried):,} carried variants')

# Verify header
result = subprocess.run(['head', '-3', 'cmt_cohort.vcf'], capture_output=True, text=True)
print(result.stdout)


In [ ]:
%%bash
set -e

WORKDIR=/opt/notebooks/vep_work
mkdir -p $WORKDIR
cd $WORKDIR

# ── 1. System dependencies ────────────────────────────────────────────────────
apt-get update -qq
apt-get install -y -qq \
    build-essential zlib1g-dev libbz2-dev liblzma-dev \
    libcurl4-openssl-dev libssl-dev cpanminus tabix

# ── 2. Perl dependencies ──────────────────────────────────────────────────────
cpanm --notest \
    Bio::Perl DBI DBD::mysql JSON \
    LWP::UserAgent LWP::Simple Archive::Zip \
    IO::Compress::Gzip Set::IntervalTree \
    PerlIO::gzip IO::Uncompress::Gunzip

# ── 3. Build htslib from source ───────────────────────────────────────────────
wget -q https://github.com/samtools/htslib/releases/download/1.19/htslib-1.19.tar.bz2
tar xjf htslib-1.19.tar.bz2
cd htslib-1.19
./configure --prefix=/usr/local
make -j4
make install
ldconfig
cd $WORKDIR
echo "htslib installed"

# ── 4. Bio::DB::HTS ───────────────────────────────────────────────────────────
HTSLIB_DIR=/usr/local cpanm --notest Bio::DB::HTS
echo "Bio::DB::HTS installed"

# ── 5. Clone VEP and Ensembl API repos ───────────────────────────────────────
git clone --depth 1 -b release/115 https://github.com/Ensembl/ensembl-vep.git
cd ensembl-vep
mkdir -p .vep
git clone --depth 1 -b release/115 https://github.com/Ensembl/ensembl.git
git clone --depth 1 -b release/115 https://github.com/Ensembl/ensembl-variation.git
git clone --depth 1 -b release/115 https://github.com/Ensembl/ensembl-funcgen.git
git clone --depth 1 -b release/115 https://github.com/Ensembl/ensembl-io.git
echo "VEP and Ensembl API repos cloned"

# ── 6. Download VEP cache ─────────────────────────────────────────────────────
curl -O https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_vep_115_GRCh38.tar.gz
tar xzf homo_sapiens_vep_115_GRCh38.tar.gz -C .vep
rm homo_sapiens_vep_115_GRCh38.tar.gz
echo "VEP cache extracted"

# ── 7. Download and index GRCh38 FASTA ───────────────────────────────────────
wget -q ftp://ftp.ensembl.org/pub/release-115/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
echo "FASTA downloaded. Decompressing..."
gunzip Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
samtools faidx Homo_sapiens.GRCh38.dna.primary_assembly.fa
echo "FASTA indexed"

# ── 8. Verify ─────────────────────────────────────────────────────────────────
export PERL5LIB=$WORKDIR/ensembl-vep/modules:\
$WORKDIR/ensembl-vep/ensembl/modules:\
$WORKDIR/ensembl-vep/ensembl-variation/modules:\
$WORKDIR/ensembl-vep/ensembl-funcgen/modules:\
$WORKDIR/ensembl-vep/ensembl-io/modules:\
/usr/local/lib/x86_64-linux-gnu/perl/5.38.2:\
/usr/local/share/perl/5.38.2:\
/usr/lib/x86_64-linux-gnu/perl5/5.38:\
/usr/share/perl5

perl -e "use Bio::Perl;              print 'BioPerl OK\n'"
perl -e "use Bio::DB::HTS::Tabix;    print 'Bio::DB::HTS OK\n'"
perl -e "use Bio::EnsEMBL::Registry; print 'Registry OK\n'"
perl -e "use DBI;                     print 'DBI OK\n'"
echo "All checks passed - ready to run VEP"


In [ ]:
%%bash
set -e

WORKDIR=/opt/notebooks/vep_work

export PERL5LIB=$WORKDIR/ensembl-vep/modules:\
$WORKDIR/ensembl-vep/ensembl/modules:\
$WORKDIR/ensembl-vep/ensembl-variation/modules:\
$WORKDIR/ensembl-vep/ensembl-funcgen/modules:\
$WORKDIR/ensembl-vep/ensembl-io/modules:\
/usr/local/lib/x86_64-linux-gnu/perl/5.38.2:\
/usr/local/share/perl/5.38.2:\
/usr/lib/x86_64-linux-gnu/perl5/5.38:\
/usr/share/perl5

cd $WORKDIR/ensembl-vep

perl ./vep \
    --cache \
    --offline \
    --dir_cache .vep \
    --assembly GRCh38 \
    --species homo_sapiens \
    --fasta ./Homo_sapiens.GRCh38.dna.primary_assembly.fa \
    --input_file /opt/notebooks/cmt_cohort.vcf \
    --output_file /opt/notebooks/cmt_cohort_anno_final.vcf \
    --vcf --pick --everything --canonical \
    --no_stats \
    --force_overwrite --fork 8 --buffer_size 5000

echo "Exit code: $?"
echo "VEP annotation complete"


---
## Stage 6 - Pathogenic Variant Filtering


In [ ]:
# Load annotated VCF — use read_vcf() to correctly skip ## meta-lines
cmt_anno = read_vcf('cmt_cohort_anno_final.vcf')
print(f'Annotated CMT cohort variants: {len(cmt_anno):,}')
print(f'Columns: {list(cmt_anno.columns[:15])}')


In [ ]:
PATHOGENIC_TERMS = {'pathogenic', 'likely_pathogenic'}
BENIGN_TERMS     = {'benign', 'likely_benign'}

def is_pathogenic(clin_sig):
    if pd.isna(clin_sig) or str(clin_sig).strip() in ('.', '', 'NA'):
        return False
    terms          = {t.strip().lower() for t in str(clin_sig).replace('&', ',').replace('/', ',').split(',')}
    has_pathogenic = bool(terms & PATHOGENIC_TERMS)
    only_benign    = terms.issubset(BENIGN_TERMS)
    return has_pathogenic and not only_benign

if 'CLIN_SIG' not in cmt_anno.columns:
    raise ValueError('CLIN_SIG column not found — check VEP annotation completed correctly')

pathogenic_variants = cmt_anno[cmt_anno['CLIN_SIG'].apply(is_pathogenic)].copy()

print(f'Total variants:                  {len(cmt_anno):>8,}')
print(f'Pathogenic / likely_pathogenic:  {len(pathogenic_variants):>8,}')
print()
print('CLIN_SIG value counts (pathogenic variants):')
print(pathogenic_variants['CLIN_SIG'].value_counts().to_string())


In [ ]:
pathogenic_variants.to_csv('pathogenic_variants.csv', index=False)
print('Saved pathogenic_variants.csv')
pathogenic_variants[['CHROM', 'POS', 'ID', 'REF', 'ALT', 'CLIN_SIG']].head(10)

---
## Stage 7 - Generate Per-Chromosome Variant Lists (Pathogenic Only)


In [ ]:
os.makedirs('variant_lists', exist_ok=True)

unique_chroms = pathogenic_variants['CHROM'].unique()
for chrom in unique_chroms:
    chrom_data = pathogenic_variants[pathogenic_variants['CHROM'] == chrom]
    df         = chrom_data[['ID']].drop_duplicates()
    outfile    = f'variant_lists/chr{chrom}.txt'
    df.to_csv(outfile, index=False, header=False)
    print(f'  chr{chrom}: {len(df)} pathogenic variants -> {outfile}')

print(f'\nWritten variant lists for {len(unique_chroms)} chromosomes')

if len(unique_chroms) == 0:
    print('WARNING: no pathogenic variants found — whole-population extraction will be empty')
    print('Check CLIN_SIG values in cmt_anno and adjust PATHOGENIC_TERMS if needed')


In [ ]:
subprocess.run(
    ['dx', 'upload', '--destination', f'{OUTPUT_FOLDER}/variant_lists/', '-r', 'variant_lists/'],
    check=True
)
print('Pathogenic variant lists uploaded to RAP.')


---
## Stage 8 - Whole-Population Extraction (Pathogenic Variants Only)

Extracts pathogenic variants from the entire 500k WGS cohort.
One SAK job per chromosome, all running in parallel.
Only chromosomes with at least one pathogenic variant are submitted.


In [ ]:
whole_pop_job_ids = []

for chrom in unique_chroms:
    variant_list_path = f'{OUTPUT_FOLDER}/variant_lists/chr{chrom}.txt'

    # Handle chrX filename
    if str(chrom).upper() == 'X':
        pfile_base = f'{WGS_PLINK_PATH}/ukb24308_cX_b0_v1'
    else:
        pfile_base = f'{WGS_PLINK_PATH}/ukb24308_c{chrom}_b0_v1'

    cmd = (
        f'plink2 --pfile "/mnt/project{pfile_base}" '
        f'--no-psam-pheno '
        f'--extract /mnt/project{variant_list_path} '
        f'--recode vcf --out chr{chrom}_whole_pop_out'
    )

    job_id = submit_sak_job(
        name          = f'CMT_WGS_WholePop_chr{chrom}',
        cmd           = cmd,
        output_folder = f'{OUTPUT_FOLDER}/whole_pop_vcfs',
        instance      = INSTANCE_TYPE
    )
    whole_pop_job_ids.append(job_id)

print(f'Submitted {len(whole_pop_job_ids)} whole-population extraction jobs')
print(f'Chromosomes with pathogenic variants: {list(unique_chroms)}')


In [ ]:
wait_for_jobs(whole_pop_job_ids)


---
## Stage 9 - Download, Combine into Single VCF, and Inspect Results

Downloads all per-chromosome whole-population VCFs and merges them into a single
`whole_population_pathogenic.vcf` containing all variant carriers across the full cohort.


In [ ]:
# Download all per-chromosome whole-population VCFs
os.makedirs('whole_pop_vcfs', exist_ok=True)
subprocess.run(
    ['dx', 'download', f'{OUTPUT_FOLDER}/whole_pop_vcfs/*.vcf', '-o', 'whole_pop_vcfs/', '-f'],
    check=True
)

whole_files = sorted(glob.glob('whole_pop_vcfs/*.vcf'))
print(f'Downloaded {len(whole_files)} whole-population VCF files')
for f in whole_files:
    print(f'  {f}')


In [ ]:
# Load and combine all per-chromosome VCFs into a single dataframe
whole_data = [read_vcf(f) for f in whole_files]
whole_vcf  = pd.concat(whole_data, ignore_index=True)

print(f'Total variants across all chromosomes: {len(whole_vcf):,}')
print(f'Unique variant IDs:                    {whole_vcf["ID"].nunique():,}')
print(f'Chromosomes present:                   {sorted(whole_vcf["CHROM"].unique())}')


In [ ]:
# Write combined output as a single valid VCF
OUTPUT_VCF = 'whole_population_pathogenic.vcf'

with open(OUTPUT_VCF, 'w') as f:
    # Write VCF meta-information headers
    f.write('##fileformat=VCFv4.2\n')
    f.write('##source=CMT_WGS_pipeline_whole_population\n')
    f.write('##filter=MAF<0.001_pathogenic_or_likely_pathogenic\n')
    # Write data with #CHROM column header
    out = whole_vcf.copy()
    out.columns = ['#CHROM' if c == 'CHROM' else c for c in out.columns]
    out.to_csv(f, sep='\t', index=False)

print(f'Written {OUTPUT_VCF}: {len(whole_vcf):,} rows')

# Verify header
result = subprocess.run(['head', '-5', OUTPUT_VCF], capture_output=True, text=True)
print(result.stdout)


In [ ]:
# Upload combined VCF to RAP
subprocess.run(
    ['dx', 'upload', OUTPUT_VCF,
     '--destination', f'{OUTPUT_FOLDER}/whole_population_pathogenic.vcf', '--brief'],
    check=True
)
print(f'Uploaded {OUTPUT_VCF} to RAP: {OUTPUT_FOLDER}/whole_population_pathogenic.vcf')
